In [ ]:
# Install (usually pre-installed in Colab)
# !pip install torch torchvision scikit-learn matplotlib pandas tqdm

import os, random, json, time, copy
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import models, transforms, datasets
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

# ---------- Reproducibility ----------
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ---------- Output dirs ----------
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
(OUT / "logs").mkdir(exist_ok=True)
(OUT / "figures").mkdir(exist_ok=True)

Device: cpu


In [ ]:
DATA_ROOT = "./data"
weights = models.ResNet18_Weights.IMAGENET1K_V1

train_full = datasets.OxfordIIITPet(root=DATA_ROOT, split="trainval",
                                    download=True, transform=None)
test_full  = datasets.OxfordIIITPet(root=DATA_ROOT, split="test",
                                    download=True, transform=None)

# Corrected: must match exactly what train_full.classes shows
selected_classes = [
    "Abyssinian", "Bengal", "Bombay", "Birman", "Sphynx",          # 5 cats
    "Beagle", "Boxer", "Chihuahua", "Pug", "Samoyed"               # 5 dogs
]

# Safety check before indexing
missing = [c for c in selected_classes if c not in train_full.classes]
assert not missing, f"Class names not found: {missing}"

class_to_idx_full = {c: i for i, c in enumerate(train_full.classes)}
selected_idx = [class_to_idx_full[c] for c in selected_classes]
old_to_new = {old: new for new, old in enumerate(selected_idx)}

print("Selected classes (new → old index):")
for new, old in old_to_new.items():
    print(f"  {new}: {train_full.classes[old]}")

def relabel(dataset):
    keep = []
    for i, (_, y) in enumerate(dataset):
        if y in old_to_new:
            keep.append((i, old_to_new[y]))
    return keep

trainval_keep = relabel(train_full)
test_keep = relabel(test_full)
print(f"Trainval kept: {len(trainval_keep)} | Test kept: {len(test_keep)}")

100%|██████████| 792M/792M [00:21<00:00, 37.5MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 18.1MB/s]


Selected classes (new → old index):
  0: Abyssinian
  5: American Bulldog
  7: American Pit Bull Terrier
  6: Basset Hound
  33: Beagle
  4: Bengal
  8: Birman
  10: Bombay
  25: Boxer
  29: British Shorthair
Trainval kept: 996 | Test kept: 985


In [ ]:
from sklearn.model_selection import train_test_split

# Build labels array for stratification
idx_all   = np.array([i for i, _ in trainval_keep])
y_all     = np.array([y for _, y in trainval_keep])

idx_train, idx_tmp, y_train, y_tmp = train_test_split(
    idx_all, y_all, test_size=0.30, stratify=y_all, random_state=SEED)

idx_val, idx_test, y_val, y_test = train_test_split(
    idx_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

print(f"Train: {len(idx_train)} | Val: {len(idx_val)} | Test(held from trainval): {len(idx_test)}")
print(f"Official test set size: {len(test_keep)}")

Train: 697 | Val: 149 | Test(held from trainval): 150
Official test set size: 985


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# WITH Augmentation (For B4)
train_tf_aug = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# WITHOUT Augmentation (For B1, B2, B3)
train_tf_noaug = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Eval transform (For Validation and Test)
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


In [ ]:
class RelabeledDataset(torch.utils.data.Dataset):
    def __init__(self, base_ds, indices, transform):
        self.base = base_ds
        self.indices = list(indices)
        self.transform = transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        real_i = self.indices[i]
        img, old_y = self.base[real_i]
        img = self.transform(img)
        return img, old_to_new[old_y]

train_ds_aug   = RelabeledDataset(train_full, idx_train, train_tf_aug)
train_ds_noaug = RelabeledDataset(train_full, idx_train, train_tf_noaug)
val_ds         = RelabeledDataset(train_full, idx_val,   eval_tf)
test_ds        = RelabeledDataset(train_full, idx_test,  eval_tf)

BATCH = 32
train_loader_aug   = DataLoader(train_ds_aug, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
train_loader_noaug = DataLoader(train_ds_noaug, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader         = DataLoader(val_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader        = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
def build_model(config, num_classes=10, pretrained=True):
    """
    config: 'B1' scratch | 'B2' freeze backbone | 'B3' layer4+fc | 'B4'/'B5' like B3
    """
    w = models.ResNet18_Weights.IMAGENET1K_V1 if (pretrained and config != "B1") else None
    model = models.resnet18(weights=w)
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    # Freeze everything first
    for p in model.parameters():
        p.requires_grad = False

    if config == "B1":
        for p in model.parameters(): p.requires_grad = True
        lr_groups = [{"params": model.parameters(), "lr": 1e-3}]

    elif config == "B2":
        for p in model.fc.parameters(): p.requires_grad = True
        lr_groups = [{"params": model.fc.parameters(), "lr": 1e-3}]

    elif config in ("B3", "B4"):
        for p in model.layer4.parameters(): p.requires_grad = True
        for p in model.fc.parameters():     p.requires_grad = True
        lr_groups = [
            {"params": model.layer4.parameters(), "lr": 1e-4},
            {"params": model.fc.parameters(),     "lr": 1e-3},
        ]

    elif config == "B5":
        # Swap architecture — EfficientNet-B0
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        w = EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model = efficientnet_b0(weights=w)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        for p in model.parameters(): p.requires_grad = False
        for p in model.features[-2:].parameters(): p.requires_grad = True
        for p in model.classifier.parameters():    p.requires_grad = True
        lr_groups = [
            {"params": model.features[-2:].parameters(), "lr": 1e-4},
            {"params": model.classifier.parameters(),     "lr": 1e-3},
        ]

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return model.to(DEVICE), lr_groups, trainable, total

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, all_preds, all_labels = 0.0, [], []
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            all_preds.append(logits.argmax(1).cpu().numpy())
            all_labels.append(y.cpu().numpy())
    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    return (total_loss / len(loader.dataset),
            accuracy_score(labels, preds),
            f1_score(labels, preds, average="macro"))


def train_model(config, model, lr_groups, train_loader, val_loader,
                max_epochs=30, patience=4):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(lr_groups, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

    history = {"epoch": [], "train_loss": [], "val_loss": [],
               "train_acc": [], "val_acc": [], "val_f1": []}
    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    t0 = time.time()

    for epoch in range(1, max_epochs + 1):
        tr_loss, tr_acc, _  = run_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc, vl_f1 = run_epoch(model, val_loader, criterion, None)
        scheduler.step()

        history["epoch"].append(epoch)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)
        history["val_f1"].append(vl_f1)

        print(f"[{config}] Ep {epoch:02d} | tr_loss {tr_loss:.4f} | "
              f"vl_loss {vl_loss:.4f} | vl_acc {vl_acc:.4f} | vl_f1 {vl_f1:.4f}")

        if vl_loss < best_val_loss - 1e-4:
            best_val_loss = vl_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    elapsed = time.time() - t0
    model.load_state_dict(best_state)
    return model, history, elapsed

In [ ]:
# Patch: report val_acc at the epoch of best val_loss
rows = []
for cfg in ["B1", "B2", "B3", "B4"]:
    r = results[cfg]
    h = r["history"]
    best_idx = int(np.argmin(h["val_loss"]))
    rows.append({
        "Condition": cfg,
        "Trainable Params": r["trainable_params"],
        "Epochs Run": r["epochs_run"],
        "Time (s)": r["train_time_sec"],
        "Val Acc @ best val_loss": round(h["val_acc"][best_idx], 4),
        "Val Loss @ best":          round(h["val_loss"][best_idx], 4),
        "Test Acc":                 round(r["test_acc"], 4),
        "Test Macro F1":            round(r["test_macro_f1"], 4),
    })

df_fixed = pd.DataFrame(rows)
df_fixed.to_csv(OUT / "logs" / "summary_fixed.csv", index=False)
df_fixed

NameError: name 'results' is not defined

In [ ]:
CONFIGS = ["B1", "B2", "B3", "B4"]
results = {}

for cfg in CONFIGS:
    print("\n" + "="*60)
    print(f"Running condition {cfg}")
    print("="*60)
    set_seed(SEED)

    # Use augmented dataloader ONLY for B4
    current_train_loader = train_loader_aug if cfg == "B4" else train_loader_noaug

    model, lr_groups, trainable, total = build_model(cfg)
    print(f"Trainable: {trainable:,} / {total:,} | Using augmentation: {cfg == 'B4'}")

    model, history, elapsed = train_model(cfg, model, lr_groups,
                                          current_train_loader, val_loader,
                                          max_epochs=30, patience=4)

    # Test-set eval
    criterion = nn.CrossEntropyLoss()
    te_loss, te_acc, te_f1 = run_epoch(model, test_loader, criterion, None)

    results[cfg] = {
        "trainable_params": trainable,
        "total_params": total,
        "epochs_run": len(history["epoch"]),
        "train_time_sec": round(elapsed, 1),
        "best_val_acc": max(history["val_acc"]),
        "test_acc": te_acc,
        "test_macro_f1": te_f1,
        "history": history,
    }

    with open(OUT / "logs" / f"{cfg}.json", "w") as f:
        json.dump(results[cfg], f, indent=2)
    torch.save(model.state_dict(), OUT / f"model_{cfg}.pt")


In [ ]:
rows = []
for cfg, r in results.items():
    rows.append({
        "Condition": cfg,
        "Trainable Params": r["trainable_params"],
        "Epochs Run": r["epochs_run"],
        "Time (s)": r["train_time_sec"],
        "Best Val Acc": round(r["best_val_acc"], 4),
        "Test Acc": round(r["test_acc"], 4),
        "Test Macro F1": round(r["test_macro_f1"], 4),
    })
df = pd.DataFrame(rows).sort_values("Condition")
df.to_csv(OUT / "logs" / "summary.csv", index=False)
df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, cfg in zip(axes, ["B1", "B2", "B3"]):
    h = results[cfg]["history"]
    ax.plot(h["epoch"], h["train_loss"], label="train")
    ax.plot(h["epoch"], h["val_loss"],   label="val")
    ax.set_title(f"{cfg} — loss"); ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout()
plt.savefig(OUT / "figures" / "loss_curves_B1_B2_B3.png", dpi=150)
plt.show()

In [ ]:
import pandas as pd

rows = []
for cfg in ["B1","B2","B3","B4"]:
    r = results[cfg]; h = r["history"]
    best = int(np.argmin(h["val_loss"]))
    rows.append({"Condition": cfg,
                 "Trainable": r["trainable_params"],
                 "Epochs": r["epochs_run"],
                 "Time(s)": r["train_time_sec"],
                 "Val Acc": round(h["val_acc"][best],4),
                 "Test Acc": round(r["test_acc"],4),
                 "Test F1": round(r["test_macro_f1"],4)})

df = pd.DataFrame(rows)
df.to_csv("summary_fixed.csv", index=False)
df

In [ ]:
# Reload best model (B3) and find misclassified test images
best_model, _, _, _ = build_model("B3")
best_model.load_state_dict(torch.load(OUT/"model_B3.pt", map_location=DEVICE))
best_model.eval()

wrong = []
with torch.no_grad():
    for x, y in test_loader:
        probs = best_model(x.to(DEVICE)).softmax(1).cpu()
        for i in range(len(y)):
            pred = probs[i].argmax().item()
            if pred != y[i].item():
                wrong.append((x[i], y[i].item(), pred, probs[i].max().item()))
        if len(wrong) >= 8:
            break

print(f"Found {len(wrong)} misclassified samples\n")
for i, (img, t, p, c) in enumerate(wrong[:6]):
    print(f"[{i}] true={selected_classes[t]:12s}  "
          f"pred={selected_classes[p]:12s}  conf={c:.3f}")

In [ ]:
import shutil, os
os.makedirs("submission", exist_ok=True)
shutil.copytree("outputs", "submission/outputs", dirs_exist_ok=True)
shutil.make_archive("Tugas05_artifacts", 'zip', "submission")

from google.colab import files
files.download("Tugas05_artifacts.zip")